# 04 — Train: Ridge regression

Searches Ridge regularization for the configured target station's direct multi-horizon water-level forecast over the joined feature artifacts, then evaluates the selected model once on the sealed test cohort.

**Inputs:** joined train/test feature artifacts and their metadata contract  
**Outputs:** in-notebook prediction preview/test metrics, an MLflow run hierarchy, and the selected model plus manifest in `models/`

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and loads the joined feature metadata. Its predictor columns are the source of truth for the model inputs: target-station engineered features plus raw measurements from every retained station at issue time `t`. The Ridge alpha search and validation policy are explicit constants so every fold and MLflow run remains inspectable.

**Parameters**

The values below are illustrative examples, not run configuration. Imported values from `src/config.py` and executable constants in this notebook are authoritative for a run; Stage-3 feature metadata is authoritative for the realized column contract, and the saved manifest records the fitted model configuration.

| Parameter | Example value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed/joined` | Directory the joined Stage-3 Parquets and metadata are read from. |
| `PREDICTION_PREVIEW_ROWS` | `5` | Number of scored test rows shown in the final preview. |
| `FULL_FEATURE_COLUMNS` | all metadata-declared predictors | The complete predictor contract used for common eligibility; raw timestamps and metadata fields are not model inputs. |
| `FEATURE_SUBSETS` | named derived subsets | Candidate feature lists built at runtime from the full metadata contract in metadata order. |
| `TARGET_COLUMNS` | `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_H` | The metadata-declared future water levels predicted directly from one issue-time feature vector. |
| `FORECAST_HORIZON_HOURS` | `24` | Example configured horizon; execution uses the imported value and validates it against the metadata contract width. |
| `RIDGE_ALPHAS` | `[0.01, 0.1, 1.0, 10.0, 100.0]` | Candidate L2 regularization strengths. |
| `LOG1P_OPTIONS` | `[False, True]` | Whether to log1p-transform the eligible water-level predictors and the target before fitting; a CV-searched axis, not a fixed choice. |
| `N_VALIDATION_FOLDS` | `5` | Number of expanding-window validation folds. |
| `INITIAL_TRAIN_FRACTION` | `0.50` | Approximate fraction of eligible rows in the first fold's training window. |
| `EMBARGO_HOURS` | `24` | Number of rows left between each fold's training and validation windows. |
| `CV_SELECTION_METRIC` | `"rmse"` | Aggregate CV metric used to select alpha; `"mae"` is also supported. |
| `MLFLOW_EXPERIMENT_NAME` | `"ridge"` | Experiment receiving the parent, nested fold, and final test runs. |
| `MODEL_PATH` | `models/ridge_{TARGET_STATION_ID}.joblib` | Bundled preprocessing and selected Ridge estimator (or `TransformedTargetRegressor` when log1p is selected) trained on all eligible training rows. |
| `MODEL_METADATA_PATH` | `models/ridge_{TARGET_STATION_ID}.json` | Schema-5.0 model-only manifest recording the estimator, preprocessor, selected subset/alpha/log1p, exact feature/target contract, and execution UUID. |

## Joint feature-subset, alpha, and log1p search

The notebook compares one global `(feature subset, alpha, log1p)` triple across `N_VALIDATION_FOLDS` expanding-window folds. Water level is a strictly positive, often right-skewed quantity, so log1p is a CV-searched axis rather than a fixed transform: when selected, it log1p-transforms the eligible water-level predictors (`water_level`, its lags, and its rolling mean/min/max — never the signed `water_level_change_*h` or the dispersion-measuring `water_level_rolling_std_*h`) plus every metadata-declared target, via a `TransformedTargetRegressor` so predictions always come back on the raw cm scale. The named subsets are derived from metadata-declared predictors and retain their metadata order:

| Subset | Intended predictors | Example size |
| --- | --- | ---: |
| `full` | All declared predictors | 81 |
| `all_station_hydrology_quality_time` | Water-level history, imputation indicators, and calendar signals for every station; excludes weather | 55 |
| `raw_all_stations` | Current `water_level`, `imputed`, precipitation, and temperature for every station | 32 |
| `target_station_full` | All declared predictors for the target station only | 53 |
| `target_station_hydrology_quality_time` | Target-station water-level history, imputation indicators, and calendar signals | 41 |
| `current_water_levels_all_stations` | Current `water_level` for every station | 8 |

The example sizes above come from one feature contract and are non-authoritative; `FEATURE_SUBSETS` built by `load_joined_dataset()` supplies the realized columns and sizes. The full contract determines eligibility once for both artifacts, so every candidate uses the same realized eligible-row counts and fold indices. A missing predictor excluded by a candidate still removes that timestamp for every candidate; smaller subsets therefore do not gain additional eligible rows in this controlled ablation. The search evaluates `len(FEATURE_SUBSETS) × len(RIDGE_ALPHAS) × len(LOG1P_OPTIONS)` candidates over `N_VALIDATION_FOLDS`, then retrains only the selected candidate and evaluates the sealed test once.

In [ ]:
import json
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import dump

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.metrics import metric_tables
from src.plots import (
    cv_error_boxplots_figure,
    model_feature_subset_candidate_distribution_figure,
    predicted_vs_actual_figure,
    test_error_boxplots_figure,
)
from src.regime_persistence import (
    regime_mlflow_metrics,
    regime_mlflow_params,
    sealed_test_regime_tables,
)
from src.training import (
    numeric_predictors,
    prediction_preview,
    summarize_cv_metrics,
    validate_predictions,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
NOTEBOOK_EXECUTION_UUID = str(uuid4())
PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
RIDGE_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
LOG1P_OPTIONS = [False, True]
MLFLOW_EXPERIMENT_NAME = "ridge"
PREDICTION_PREVIEW_ROWS = 5
if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")
station_id = TARGET_STATION_ID
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / f"ridge_{station_id}.joblib"
MODEL_METADATA_PATH = MODEL_DIR / f"ridge_{station_id}.json"

## Shared evaluation cohort

The model is fit and scored on rows from the joined feature artifacts. One row is one timestamp `t`, and it qualifies only when both conditions hold:

1. **Stage 3 marked the future window valid.** `{TARGET_STATION_ID}__target_valid` is true, and every column in `TARGET_COLUMNS` is present.
2. **Every model input is present.** All full-contract predictors must be available: the target station's engineered features plus every retained station's raw water level, imputation flag, precipitation, and temperature at issue time `t`.

Train and test are filtered independently and are never pooled: the test artifact is sealed, and no statistic used by the model — not even a scaler mean — is ever computed from it. Eligibility is deliberately based on `FULL_FEATURE_COLUMNS`, not a candidate subset, so every candidate compares the same cohort.

## Shared helpers

The joined dataset — contract loading, common-cohort preparation, ordered feature subsets, and chronological folds — comes from `src.dataset`. Prediction checks, metric summaries, and previews come from `src.training`, and the evaluation figures from `src.plots`. Ridge candidate ranking remains local because its subset/alpha tie-breaking policy is estimator-specific.

In [ ]:
from src.ridge import build_ridge_estimator, save_ridge_manifest, select_candidate

## Load the joined dataset

`load_joined_dataset()` does the whole preamble in one call: it reads the joined feature metadata, `all_stations_train_features.parquet`, and `all_stations_test_features.parquet` from the Stage-3 directory and checks their station, horizon, and column contracts, so a missing or incompatible artifact fails before any model work begins.

It then applies the eligibility cohort to each artifact independently — keeping only target-valid rows with complete predictors and targets, sorted chronologically — derives the named, ordered feature subsets, and builds the configured expanding validation folds. If either split has no eligible row, or the folds violate the configured CV policy, the notebook stops here rather than fitting on an empty frame or reporting a metric computed from nothing.

In [ ]:
dataset = load_joined_dataset(
    METADATA_PATH,
    train_path,
    test_path,
    station_id=station_id,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
contract = dataset.contract
TARGET_COLUMNS = list(contract.target_columns)
FULL_FEATURE_COLUMNS = list(contract.predictor_columns)
FEATURE_SUBSETS = dataset.feature_subsets
train_rows = dataset.train_rows
test_rows = dataset.test_rows
INPUT_PARQUET_SHA256_PARAMS = dataset.input_hashes

## Joint time-series subset, alpha, and log1p search

Eligible training rows are sorted by issue time before `TimeSeriesSplit` creates `N_VALIDATION_FOLDS` expanding-window folds. The explicit `test_size` allocates the post-initial-training portion across the folds, while `EMBARGO_HOURS` supplies the hourly embargo. Each `(subset, alpha, log1p)` candidate has one MLflow parent and each fold has one nested child run. The execution must produce the complete `len(FEATURE_SUBSETS) × len(RIDGE_ALPHAS) × len(LOG1P_OPTIONS)` Cartesian product before selection, for that candidate count multiplied by `N_VALIDATION_FOLDS` fold fits.

Every fold builds its own preprocessing-plus-Ridge estimator via `build_ridge_estimator` — a `StandardScaler` pipeline, additionally log1p-transforming the eligible water-level columns and the target when the candidate's `log1p` flag is set — using only that fold's training rows and the candidate's explicit columns. The sealed test cohort is not referenced until the final fit below.

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
cv_splits = dataset.folds
validation_test_size = dataset.validation_test_size
cv_results_rows = []
cv_horizon_rows_by_candidate = {}
expected_candidate_keys = {
    (subset_name, float(alpha), bool(log1p_flag))
    for subset_name in FEATURE_SUBSETS
    for alpha in RIDGE_ALPHAS
    for log1p_flag in LOG1P_OPTIONS
}

for subset_name, feature_columns in FEATURE_SUBSETS.items():
    for alpha in RIDGE_ALPHAS:
        for log1p_flag in LOG1P_OPTIONS:
            fold_aggregate_rows = []
            fold_horizon_rows = []
            with mlflow.start_run(
                run_name=f"ridge_cv_{subset_name}_{alpha:g}_log1p_{log1p_flag}",
                nested=False,
                tags={
                    "phase": "cv",
                    "run_type": "candidate_parent",
                    "subset": subset_name,
                    "log1p": str(log1p_flag),
                    "execution_uuid": NOTEBOOK_EXECUTION_UUID,
                },
            ):
                mlflow.log_params(
                    {
                        "phase": "cv",
                        "run_type": "candidate_parent",
                        "subset": subset_name,
                        "feature_count": len(feature_columns),
                        "feature_columns": json.dumps(feature_columns),
                        **INPUT_PARQUET_SHA256_PARAMS,
                        "alpha": alpha,
                        "log1p": log1p_flag,
                        "n_validation_folds": N_VALIDATION_FOLDS,
                        "validation_test_size": validation_test_size,
                        "embargo_hours": EMBARGO_HOURS,
                        "selection_metric": CV_SELECTION_METRIC,
                        "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                        "common_train_rows": len(train_rows),
                    }
                )

                for fold_number, (
                    fold_train_indices,
                    fold_validation_indices,
                ) in enumerate(cv_splits, start=1):
                    with mlflow.start_run(
                        run_name=f"ridge_cv_{subset_name}_{alpha:g}_log1p_{log1p_flag}_fold_{fold_number}",
                        nested=True,
                        tags={
                            "phase": "cv",
                            "run_type": "fold",
                            "subset": subset_name,
                            "log1p": str(log1p_flag),
                            "fold": str(fold_number),
                            "execution_uuid": NOTEBOOK_EXECUTION_UUID,
                        },
                    ):
                        fold_train_rows = train_rows.iloc[fold_train_indices]
                        fold_validation_rows = train_rows.iloc[fold_validation_indices]
                        fold_model = build_ridge_estimator(
                            feature_columns, alpha=alpha, log1p=log1p_flag
                        )
                        fold_model.fit(
                            numeric_predictors(fold_train_rows, feature_columns),
                            fold_train_rows[TARGET_COLUMNS],
                        )
                        fold_predictions = validate_predictions(
                            fold_model.predict(
                                numeric_predictors(
                                    fold_validation_rows, feature_columns
                                )
                            ),
                            expected_rows=len(fold_validation_rows),
                            target_columns=TARGET_COLUMNS,
                            artifact_name="fold",
                        )

                        fold_aggregate, fold_per_horizon = metric_tables(
                            fold_validation_rows[TARGET_COLUMNS],
                            fold_predictions,
                            target_columns=TARGET_COLUMNS,
                            station_id=station_id,
                        )
                        fold_aggregate_rows.append(fold_aggregate.iloc[0])
                        fold_horizon_rows.append(fold_per_horizon)
                        mlflow.log_params(
                            {
                                "phase": "cv",
                                "run_type": "fold",
                                "subset": subset_name,
                                "feature_count": len(feature_columns),
                                **INPUT_PARQUET_SHA256_PARAMS,
                                "alpha": alpha,
                                "log1p": log1p_flag,
                                "fold": fold_number,
                                "train_rows": len(fold_train_rows),
                                "validation_rows": len(fold_validation_rows),
                                "gap_rows": EMBARGO_HOURS,
                                "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                                "train_start": fold_train_rows["timestamp"]
                                .iloc[0]
                                .isoformat(),
                                "train_end": fold_train_rows["timestamp"]
                                .iloc[-1]
                                .isoformat(),
                                "validation_start": fold_validation_rows["timestamp"]
                                .iloc[0]
                                .isoformat(),
                                "validation_end": fold_validation_rows["timestamp"]
                                .iloc[-1]
                                .isoformat(),
                                "train_index_start": int(fold_train_indices[0]),
                                "train_index_end": int(fold_train_indices[-1]),
                                "validation_index_start": int(
                                    fold_validation_indices[0]
                                ),
                                "validation_index_end": int(
                                    fold_validation_indices[-1]
                                ),
                            }
                        )
                        mlflow.log_metrics(
                            {
                                "fold_mae": float(fold_aggregate.iloc[0]["mae"]),
                                "fold_rmse": float(fold_aggregate.iloc[0]["rmse"]),
                                "fold_me": float(fold_aggregate.iloc[0]["me"]),
                                "fold_r2": float(fold_aggregate.iloc[0]["r2"]),
                                **{
                                    f"fold_mae_horizon_{row.horizon_hours:02d}": float(
                                        row.mae
                                    )
                                    for row in fold_per_horizon.itertuples()
                                },
                                **{
                                    f"fold_me_horizon_{row.horizon_hours:02d}": float(
                                        row.me
                                    )
                                    for row in fold_per_horizon.itertuples()
                                },
                                **{
                                    f"fold_r2_horizon_{row.horizon_hours:02d}": float(
                                        row.r2
                                    )
                                    for row in fold_per_horizon.itertuples()
                                },
                                **{
                                    f"fold_rmse_horizon_{row.horizon_hours:02d}": float(
                                        row.rmse
                                    )
                                    for row in fold_per_horizon.itertuples()
                                },
                            }
                        )

                fold_aggregate_metrics = pd.DataFrame(fold_aggregate_rows)
                fold_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
                parent_metrics = summarize_cv_metrics(
                    fold_aggregate_metrics,
                    fold_horizon_metrics,
                )
                candidate_key = (subset_name, float(alpha), bool(log1p_flag))
                cv_horizon_rows_by_candidate[candidate_key] = fold_horizon_rows.copy()
                mlflow.log_metrics(parent_metrics)
                cv_results_rows.append(
                    {
                        "subset": subset_name,
                        "feature_count": len(feature_columns),
                        "alpha": float(alpha),
                        "log1p": bool(log1p_flag),
                        "mae_mean": parent_metrics["cv_mae_mean"],
                        "mae_std": parent_metrics["cv_mae_std"],
                        "rmse_mean": parent_metrics["cv_rmse_mean"],
                        "rmse_std": parent_metrics["cv_rmse_std"],
                        "me_mean": parent_metrics["cv_me_mean"],
                        "me_std": parent_metrics["cv_me_std"],
                        "r2_mean": parent_metrics["cv_r2_mean"],
                        "r2_std": parent_metrics["cv_r2_std"],
                        **{
                            metric_name: metric_value
                            for metric_name, metric_value in parent_metrics.items()
                            if metric_name
                            not in {
                                "cv_mae_mean",
                                "cv_mae_std",
                                "cv_rmse_mean",
                                "cv_rmse_std",
                                "cv_me_mean",
                                "cv_me_std",
                                "cv_r2_mean",
                                "cv_r2_std",
                            }
                        },
                    }
                )

cv_experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if cv_experiment is None:
    raise ValueError(f"MLflow experiment {MLFLOW_EXPERIMENT_NAME!r} was not found")
current_cv_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'candidate_parent'"
    ),
)
current_fold_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'fold'"
    ),
)
parent_keys = {
    (
        str(row["tags.subset"]),
        float(row["params.alpha"]),
        row["tags.log1p"] == "True",
    )
    for _, row in current_cv_runs.iterrows()
}
if (
    len(current_cv_runs) != len(expected_candidate_keys)
    or parent_keys != expected_candidate_keys
):
    raise ValueError(
        "Current execution must produce the complete 60-candidate subset/alpha/log1p product: "
        f"expected {len(expected_candidate_keys)} {sorted(expected_candidate_keys)}, "
        f"got {len(current_cv_runs)} {sorted(parent_keys)}"
    )
expected_fold_count = len(expected_candidate_keys) * N_VALIDATION_FOLDS
if len(current_fold_runs) != expected_fold_count:
    raise ValueError(
        f"Current execution must produce {expected_fold_count} nested fold runs, got {len(current_fold_runs)}"
    )
fold_keys = {
    (
        str(row["tags.subset"]),
        float(row["params.alpha"]),
        row["tags.log1p"] == "True",
        int(row["tags.fold"]),
    )
    for _, row in current_fold_runs.iterrows()
}
expected_fold_keys = {
    (subset_name, alpha, log1p_flag, fold_number)
    for subset_name, alpha, log1p_flag in expected_candidate_keys
    for fold_number in range(1, N_VALIDATION_FOLDS + 1)
}
if fold_keys != expected_fold_keys:
    raise ValueError(
        "Current execution fold runs do not cover every candidate and fold"
    )
cv_results = pd.DataFrame(cv_results_rows)
if len(cv_results) != len(expected_candidate_keys):
    raise ValueError("The in-memory CV result table is incomplete")
if (
    set(zip(cv_results["subset"], cv_results["alpha"], cv_results["log1p"]))
    != expected_candidate_keys
):
    raise ValueError(
        "The in-memory CV result table does not match the candidate product"
    )
cv_results = cv_results.sort_values(
    ["subset", "alpha", "log1p"], kind="stable"
).reset_index(drop=True)
selected_subset, selected_alpha, selected_log1p = select_candidate(
    cv_results, CV_SELECTION_METRIC
)
selected_feature_columns = FEATURE_SUBSETS[selected_subset]
fold_horizon_rows = cv_horizon_rows_by_candidate[
    (selected_subset, selected_alpha, selected_log1p)
]
print(
    f"Selected Ridge candidate by CV {CV_SELECTION_METRIC.upper()}: "
    f"{selected_subset!r}, alpha={selected_alpha:g}, log1p={selected_log1p}"
)
display(
    cv_results[
        [
            "subset",
            "feature_count",
            "alpha",
            "log1p",
            "mae_mean",
            "mae_std",
            "rmse_mean",
            "rmse_std",
            "me_mean",
            "me_std",
            "r2_mean",
            "r2_std",
        ]
    ]
)

## Retrain the selected subset, alpha, and log1p

The selected `(feature subset, alpha, log1p)` triple is retrained once on all eligible, chronologically ordered training rows, using the same `build_ridge_estimator` construction path as every CV fold. The estimator is fitted on the full eligible training cohort, then the sealed test predictors are scored. The model is held in memory here; it is written to disk together with its manifest only after the sealed-test cell below succeeds, so a crashed run leaves the previous artifacts intact.

In [ ]:
final_model = build_ridge_estimator(
    selected_feature_columns, alpha=selected_alpha, log1p=selected_log1p
)
final_model.fit(
    numeric_predictors(train_rows, selected_feature_columns),
    train_rows[TARGET_COLUMNS],
)
test_predictions = validate_predictions(
    final_model.predict(numeric_predictors(test_rows, selected_feature_columns)),
    expected_rows=len(test_rows),
    target_columns=TARGET_COLUMNS,
    artifact_name="test",
)
print(
    f"Fitted the selected Ridge candidate on {len(train_rows):,} eligible training "
    f"rows and scored {len(test_rows):,} sealed-test rows."
)

## Evaluate on the test cohort

A single scoring pass over the sealed test cohort reports aggregate MAE/RMSE, the same metrics for each configured lead in the direct forecast, and a short preview for comparison with actual targets. Plot and MLflow labels identify the selected subset, alpha, and log1p flag. There is no second pass and no refitting.

The model and its model-only manifest are written at the end of this cell. The manifest contains the estimator configuration and exact feature/target contract; CV and sealed-test diagnostics are logged to MLflow and remain available in memory for the analysis below.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS],
    test_predictions,
    target_columns=TARGET_COLUMNS,
    station_id=station_id,
)
regime_definition, regime_aggregate_metrics, regime_horizon_metrics = (
    sealed_test_regime_tables(
        test_rows[TARGET_COLUMNS],
        test_predictions,
        target_columns=TARGET_COLUMNS,
        station_id=station_id,
        quartile_cutoffs_cm=dataset.target_water_level_quartile_cutoffs_cm,
        quartile_reference_count=dataset.target_water_level_quartile_reference_count,
    )
)
if not np.isfinite(aggregate_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all():
    raise ValueError("Ridge reported non-finite aggregate metrics")
if not np.isfinite(per_horizon_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all():
    raise ValueError("Ridge reported non-finite horizon metrics")
with mlflow.start_run(
    run_name=f"ridge_test_{selected_subset}_alpha_{selected_alpha:g}",
    nested=False,
    tags={
        "phase": "test",
        "run_type": "sealed_test",
        "subset": selected_subset,
        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    },
):
    mlflow.log_params(
        {
            "phase": "test",
            "run_type": "sealed_test",
            "subset": selected_subset,
            "feature_count": len(selected_feature_columns),
            "feature_columns": json.dumps(selected_feature_columns),
            **INPUT_PARQUET_SHA256_PARAMS,
            "alpha": selected_alpha,
            "log1p": str(selected_log1p),
            "selection_metric": CV_SELECTION_METRIC,
            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
            "cv_selected_metric": float(
                cv_results.loc[
                    cv_results["subset"].eq(selected_subset)
                    & cv_results["alpha"].eq(selected_alpha)
                    & cv_results["log1p"].eq(selected_log1p),
                    f"{CV_SELECTION_METRIC}_mean",
                ].iloc[0]
            ),
            **regime_mlflow_params(regime_definition),
            "scored_issue_times": len(test_rows),
        }
    )
    mlflow.log_metrics(
        {
            **regime_mlflow_metrics(regime_aggregate_metrics, regime_horizon_metrics),
            "test_mae": float(aggregate_metrics.iloc[0]["mae"]),
            "test_rmse": float(aggregate_metrics.iloc[0]["rmse"]),
            "test_me": float(aggregate_metrics.iloc[0]["me"]),
            "test_r2": float(aggregate_metrics.iloc[0]["r2"]),
            **{
                f"test_mae_horizon_{row.horizon_hours:02d}": float(row.mae)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_me_horizon_{row.horizon_hours:02d}": float(row.me)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_r2_horizon_{row.horizon_hours:02d}": float(row.r2)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
                for row in per_horizon_metrics.itertuples()
            },
        }
    )
    cv_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
    cv_rmse_mae_boxplots_fig = cv_error_boxplots_figure(
        cv_horizon_metrics,
        TARGET_COLUMNS,
        title=f"Ridge CV errors — {selected_subset}, alpha={selected_alpha:g}, log1p={selected_log1p}",
    )
    mlflow.log_figure(cv_rmse_mae_boxplots_fig, "cv_rmse_mae_boxplots.png")
    plt.show()
    plt.close(cv_rmse_mae_boxplots_fig)
    test_error_boxplots_fig = test_error_boxplots_figure(
        test_rows,
        test_predictions,
        per_horizon_metrics,
        TARGET_COLUMNS,
        title=f"Ridge final-test errors — {selected_subset}, alpha={selected_alpha:g}, log1p={selected_log1p}",
    )
    mlflow.log_figure(test_error_boxplots_fig, "test_error_boxplots.png")
    plt.show()
    plt.close(test_error_boxplots_fig)
    test_predicted_vs_actual_fig = predicted_vs_actual_figure(
        test_rows[TARGET_COLUMNS],
        test_predictions,
        TARGET_COLUMNS,
        title=f"Ridge predicted vs actual — {selected_subset}, alpha={selected_alpha:g}, log1p={selected_log1p}",
    )
    mlflow.log_figure(test_predicted_vs_actual_fig, "test_predicted_vs_actual.png")
    plt.show()
    plt.close(test_predicted_vs_actual_fig)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
dump(final_model, MODEL_PATH)
save_ridge_manifest(
    MODEL_METADATA_PATH,
    model_path=MODEL_PATH,
    execution_uuid=NOTEBOOK_EXECUTION_UUID,
    contract=contract,
    feature_subsets=FEATURE_SUBSETS,
    selected_subset=selected_subset,
    selected_alpha=selected_alpha,
    selected_log1p=selected_log1p,
)
print(f"Saved Ridge model to {MODEL_PATH}")
print(f"Saved Ridge model manifest to {MODEL_METADATA_PATH}")
print(
    f"Ridge test results for {station_id} "
    f"(selected subset={selected_subset!r}, alpha={selected_alpha:g}, log1p={selected_log1p})"
)
display(aggregate_metrics)
display(per_horizon_metrics)
display(
    prediction_preview(
        test_rows,
        test_predictions,
        target_columns=TARGET_COLUMNS,
    ).head(PREDICTION_PREVIEW_ROWS)
)

# Ridge saved-model evaluation

This section validates the saved Ridge model manifest and presents the current execution's in-memory CV and sealed-test diagnostics. Results are stored in MLflow, not in the model manifest.


## Load the saved Ridge execution record

The joined dataset and model manifest are loaded here to validate the saved model against the current station, horizon, target columns, and selected subset columns. The comparison tables also use the current execution's in-memory `cv_results`, `aggregate_metrics`, and `per_horizon_metrics`; results are deliberately not read from the manifest.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.plots import forecast_window_figures
from src.ridge import load_ridge_manifest, score_saved_model

if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")

COMPARISON_PROCESSED_DIR = Path("data/processed/joined")
COMPARISON_METADATA_PATH = (
    COMPARISON_PROCESSED_DIR / "all_stations_feature_metadata.json"
)
COMPARISON_TRAIN_PATH = COMPARISON_PROCESSED_DIR / "all_stations_train_features.parquet"
COMPARISON_TEST_PATH = COMPARISON_PROCESSED_DIR / "all_stations_test_features.parquet"
COMPARISON_MODEL_PATH = Path("models") / f"ridge_{TARGET_STATION_ID}.joblib"
COMPARISON_MODEL_METADATA_PATH = Path("models") / f"ridge_{TARGET_STATION_ID}.json"

comparison_dataset = load_joined_dataset(
    COMPARISON_METADATA_PATH,
    COMPARISON_TRAIN_PATH,
    COMPARISON_TEST_PATH,
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
comparison_contract = comparison_dataset.contract
COMPARISON_TARGET_COLUMNS = list(comparison_contract.target_columns)
comparison_test_rows = comparison_dataset.test_rows

ridge_manifest = load_ridge_manifest(
    COMPARISON_MODEL_METADATA_PATH,
    contract=comparison_contract,
    feature_subsets=comparison_dataset.feature_subsets,
)

## Inspect the recorded execution

The manifest is written once, after the sealed test has been scored, so an execution that crashed part-way leaves no record and the previous manifest survives untouched. The `execution_uuid` below is the same tag the MLflow runs of that execution carry, which is how the two views are tied together.


In [ ]:
selected_candidate_table = cv_results
selected_horizon_metrics = per_horizon_metrics.rename(
    columns={metric: f"test_{metric}" for metric in ("mae", "rmse", "me", "r2")}
)
selected_candidate = selected_candidate_table.loc[
    selected_candidate_table["subset"].eq(ridge_manifest.selected_subset)
    & selected_candidate_table["alpha"].eq(ridge_manifest.selected_alpha)
    & selected_candidate_table["log1p"].eq(ridge_manifest.selected_log1p)
].iloc[0]
selected_execution_summary = pd.DataFrame(
    [
        {
            "execution_uuid": ridge_manifest.execution_uuid,
            "manifest": str(COMPARISON_MODEL_METADATA_PATH),
            "candidate_count": len(selected_candidate_table),
            "selection_metric": CV_SELECTION_METRIC,
            "selected_subset": ridge_manifest.selected_subset,
            "selected_alpha": ridge_manifest.selected_alpha,
            "selected_log1p": ridge_manifest.selected_log1p,
        }
    ]
)
display(selected_execution_summary)

## Compare cross-validation candidates

Candidate ranking uses only the recorded CV metrics and the configured selection metric; the sealed-test metrics are not used to rank candidates.


In [ ]:
candidate_columns = [
    "subset",
    "feature_count",
    "alpha",
    "log1p",
    "mae_mean",
    "mae_std",
    "rmse_mean",
    "rmse_std",
    "me_mean",
    "me_std",
    "r2_mean",
    "r2_std",
]
candidate_comparison_table = selected_candidate_table[candidate_columns].copy()
display(candidate_comparison_table)

### Compare performance across feature subsets

Each point is one hyperparameter candidate's aggregate CV mean. The boxes summarize the candidate distribution within each feature subset; they do not show fold-to-fold uncertainty.

In [ ]:
candidate_distribution_figure = model_feature_subset_candidate_distribution_figure(
    candidate_comparison_table,
    model_name="Ridge",
    hover_columns=("feature_count", "alpha", "log1p"),
)
display(candidate_distribution_figure)

## Visualize cross-validation error

The heatmap shows CV RMSE across feature subsets and alpha values. The line chart adds fold-to-fold RMSE variation as error bars.


In [ ]:
LOG1P_FACETS = [False, True]
cv_rmse_heatmap_zmin = candidate_comparison_table["rmse_mean"].min()
cv_rmse_heatmap_zmax = candidate_comparison_table["rmse_mean"].max()
cv_rmse_heatmap_figure = make_subplots(
    rows=1,
    cols=len(LOG1P_FACETS),
    subplot_titles=[f"log1p={log1p_flag}" for log1p_flag in LOG1P_FACETS],
    shared_yaxes=True,
)
for column_index, log1p_flag in enumerate(LOG1P_FACETS, start=1):
    facet_table = candidate_comparison_table[
        candidate_comparison_table["log1p"].eq(log1p_flag)
    ]
    facet_heatmap_values = (
        facet_table.pivot(index="subset", columns="alpha", values="rmse_mean")
        .sort_index(axis=0)
        .sort_index(axis=1)
    )
    cv_rmse_heatmap_figure.add_trace(
        go.Heatmap(
            z=facet_heatmap_values.to_numpy(),
            x=list(map(str, facet_heatmap_values.columns.tolist())),
            y=facet_heatmap_values.index.tolist(),
            zmin=cv_rmse_heatmap_zmin,
            zmax=cv_rmse_heatmap_zmax,
            coloraxis="coloraxis",
            hovertemplate="Subset=%{y}<br>Alpha=%{x}<br>CV RMSE=%{z:.4f}<extra></extra>",
        ),
        row=1,
        col=column_index,
    )
cv_rmse_heatmap_figure.update_layout(
    title="Ridge candidate CV RMSE by feature subset, alpha, and log1p",
    coloraxis={"colorscale": "Viridis", "colorbar": {"title": "CV RMSE"}},
)
cv_rmse_heatmap_figure.update_xaxes(title_text="Alpha")
cv_rmse_heatmap_figure.update_yaxes(title_text="Feature subset", col=1)
display(cv_rmse_heatmap_figure)

In [ ]:
cv_rmse_by_alpha_figure = go.Figure()
for (subset_name, log1p_flag), subset_candidates in candidate_comparison_table.groupby(
    ["subset", "log1p"], sort=True
):
    subset_candidates = subset_candidates.sort_values("alpha")
    cv_rmse_by_alpha_figure.add_trace(
        go.Scatter(
            x=subset_candidates["alpha"],
            y=subset_candidates["rmse_mean"],
            mode="lines+markers",
            name=f"{subset_name} · log1p={log1p_flag}",
            line={"dash": "dash" if log1p_flag else "solid"},
            error_y={
                "type": "data",
                "array": subset_candidates["rmse_std"],
                "visible": True,
            },
        )
    )
cv_rmse_by_alpha_figure.update_layout(
    title="Ridge CV RMSE versus alpha, by feature subset and log1p",
    xaxis_title="Alpha",
    yaxis_title="CV RMSE",
    xaxis_type="log",
)
display(cv_rmse_by_alpha_figure)

## Inspect selected-candidate sealed-test performance

These values belong only to the candidate recorded by the selected sealed-test run. The final chart shows its MAE and RMSE at each available forecast horizon.


In [ ]:
selected_candidate_sealed_test_summary = pd.DataFrame(
    [
        {
            "execution_uuid": ridge_manifest.execution_uuid,
            "subset": selected_candidate["subset"],
            "feature_count": selected_candidate["feature_count"],
            "alpha": selected_candidate["alpha"],
            "log1p": selected_candidate["log1p"],
            "cv_mae_mean": selected_candidate["mae_mean"],
            "cv_rmse_mean": selected_candidate["rmse_mean"],
            "cv_me_mean": selected_candidate["me_mean"],
            "cv_r2_mean": selected_candidate["r2_mean"],
            **{
                f"test_{metric}": float(aggregate_metrics.iloc[0][metric])
                for metric in ("mae", "rmse", "me", "r2")
            },
        }
    ]
)
display(selected_candidate_sealed_test_summary)

sealed_test_horizon_figure = go.Figure(
    [
        go.Scatter(
            x=selected_horizon_metrics["horizon_hours"],
            y=selected_horizon_metrics[metric_name],
            mode="lines+markers",
            name=metric_name.upper(),
        )
        for metric_name in ("test_mae", "test_rmse", "test_me", "test_r2")
    ]
)
sealed_test_horizon_figure.update_layout(
    title="Selected Ridge candidate sealed-test error by horizon",
    xaxis_title="Forecast horizon (hours)",
    yaxis_title="Error",
)
display(sealed_test_horizon_figure)

## Reload and score the saved Ridge model

Reloads the saved Ridge model and scores the eligible sealed-test cohort on the manifest's selected feature columns, without retraining or changing the stored prediction semantics.


In [ ]:
comparison_prediction_values = score_saved_model(
    ridge_manifest,
    COMPARISON_MODEL_PATH,
    comparison_test_rows,
)
print(
    f"Scored {len(comparison_test_rows):,} eligible sealed-test rows with "
    f"{len(COMPARISON_TARGET_COLUMNS)} horizons using the saved Ridge model."
)

In [ ]:
comparison_prediction_columns = [
    f"prediction_{target_column}" for target_column in COMPARISON_TARGET_COLUMNS
]
comparison_prediction_table = (
    comparison_test_rows[["timestamp", *COMPARISON_TARGET_COLUMNS]]
    .reset_index(drop=True)
    .rename(columns={"timestamp": "issue_time"})
)
comparison_prediction_table = pd.concat(
    [
        comparison_prediction_table,
        pd.DataFrame(
            comparison_prediction_values,
            columns=comparison_prediction_columns,
        ),
    ],
    axis=1,
)
comparison_prediction_table["issue_time"] = pd.to_datetime(
    comparison_prediction_table["issue_time"], utc=True
)

comparison_issue_times = comparison_prediction_table["issue_time"]
comparison_horizons = list(range(1, FORECAST_HORIZON_HOURS + 1))
comparison_horizon_labels = [f"H+{horizon:02d}" for horizon in comparison_horizons]
comparison_time_series_frames = []
for horizon in comparison_horizons:
    target_column = COMPARISON_TARGET_COLUMNS[horizon - 1]
    prediction_column = comparison_prediction_columns[horizon - 1]
    valid_times = comparison_issue_times + pd.to_timedelta(horizon, unit="h")
    comparison_time_series_frames.append(
        go.Frame(
            name=comparison_horizon_labels[horizon - 1],
            data=[
                go.Scattergl(
                    x=valid_times,
                    y=comparison_prediction_table[target_column],
                    customdata=comparison_issue_times,
                    mode="lines+markers",
                    name="Actual",
                    hovertemplate="Valid time=%{x}<br>Issue time=%{customdata}<br>Actual=%{y:.3f}<extra></extra>",
                ),
                go.Scattergl(
                    x=valid_times,
                    y=comparison_prediction_table[prediction_column],
                    customdata=comparison_issue_times,
                    mode="lines+markers",
                    name="Prediction",
                    hovertemplate="Valid time=%{x}<br>Issue time=%{customdata}<br>Prediction=%{y:.3f}<extra></extra>",
                ),
            ],
        )
    )
comparison_time_series_steps = [
    {
        "label": comparison_horizon_labels[horizon - 1],
        "method": "animate",
        "args": [[comparison_horizon_labels[horizon - 1]], {"mode": "immediate"}],
    }
    for horizon in comparison_horizons
]
comparison_time_series_figure = go.Figure(
    data=comparison_time_series_frames[0].data,
    frames=comparison_time_series_frames,
    layout={
        "title": "Saved Ridge predictions across forecast horizons",
        "xaxis_title": "Valid time",
        "yaxis_title": "Water level",
        "hovermode": "x unified",
        "sliders": [
            {
                "active": 0,
                "currentvalue": {"prefix": "Forecast horizon: "},
                "steps": comparison_time_series_steps,
            }
        ],
    },
)
ridge_prediction_time_series_figure = comparison_time_series_figure
display(comparison_time_series_figure)

## Inspect best and worst Ridge forecast windows

The following plots use the saved-model predictions and select sealed-test issue times by the RMSE calculated across every configured forecast horizon. A context window is eligible only when the target-station water-level series contains every hourly observation across the notebook's configured context window, with no imputed observations.

In [ ]:
ridge_forecast_window_figures = forecast_window_figures(
    comparison_prediction_table,
    comparison_dataset.target_context_series,
    water_level_column=f"{TARGET_STATION_ID}__water_level",
    imputed_column=f"{TARGET_STATION_ID}__imputed",
    prediction_columns=comparison_prediction_columns,
    target_columns=COMPARISON_TARGET_COLUMNS,
    horizons=comparison_horizons,
    label_prefix="Ridge",
)

### Best 

In [ ]:
best_ridge_forecast_window_figure = ridge_forecast_window_figures["best"]
display(best_ridge_forecast_window_figure)

### Worst

In [ ]:
worst_ridge_forecast_window_figure = ridge_forecast_window_figures["worst"]
display(worst_ridge_forecast_window_figure)

## Compare absolute and signed errors

Each box contains all eligible sealed-test errors for one horizon. Absolute-error markers reuse the current execution's per-horizon MAE/RMSE values; signed errors follow the convention `prediction - actual`.


In [ ]:
comparison_actual_values = comparison_test_rows[COMPARISON_TARGET_COLUMNS].to_numpy(
    dtype=float
)
signed_errors = comparison_prediction_values - comparison_actual_values
absolute_errors = np.abs(signed_errors)

absolute_error_boxplot_figure = go.Figure(
    data=[
        go.Box(
            x=comparison_horizon_labels * len(absolute_errors),
            y=absolute_errors.reshape(-1),
            name="Boxplots",
            boxpoints=False,
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=selected_horizon_metrics["test_mae"],
            mode="markers",
            name="MAE",
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=selected_horizon_metrics["test_rmse"],
            mode="markers",
            name="RMSE",
        ),
    ]
)
absolute_error_boxplot_figure.update_layout(
    title="Saved Ridge absolute errors by forecast horizon",
    xaxis={
        "title": "Forecast horizon",
        "type": "category",
        "categoryorder": "array",
        "categoryarray": comparison_horizon_labels,
    },
    yaxis_title="Absolute error",
)

display(absolute_error_boxplot_figure)

In [ ]:
signed_error_boxplot_figure = go.Figure(
    data=[
        go.Box(
            x=comparison_horizon_labels * len(signed_errors),
            y=signed_errors.reshape(-1),
            name="Boxplots",
            boxpoints=False,
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=signed_errors.mean(axis=0),
            mode="markers",
            name="Mean error",
        ),
    ]
)
signed_error_boxplot_figure.update_layout(
    title="Saved Ridge signed errors by forecast horizon",
    xaxis={
        "title": "Forecast horizon",
        "type": "category",
        "categoryorder": "array",
        "categoryarray": comparison_horizon_labels,
    },
    yaxis_title="Signed error (prediction - actual)",
)
signed_error_boxplot_figure.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
)
display(signed_error_boxplot_figure)

In [ ]:
from IPython.display import Markdown, display
from joblib import load as load_joblib

formula_model = load_joblib(COMPARISON_MODEL_PATH)
formula_regressor = getattr(formula_model, "regressor_", formula_model)
formula_preprocessor = formula_regressor.named_steps["preprocess"]
formula_ridge = formula_regressor.named_steps["ridge"]

formula_feature_rows = []
for transformer_name, transformer, columns in formula_preprocessor.transformers_:
    if isinstance(transformer, str):
        continue
    scaler = (
        transformer.named_steps["scale"]
        if hasattr(transformer, "named_steps")
        else transformer
    )
    input_transform = (
        "log1p(x) = log(1 + x)" if transformer_name == "log1p_scale" else "x"
    )
    formula_feature_rows.extend(
        {
            "feature": feature,
            "input_transform": input_transform,
            "center_mu": mean,
            "scale_s": scale,
        }
        for feature, mean, scale in zip(
            list(columns), scaler.mean_, scaler.scale_, strict=True
        )
    )
formula_feature_table = pd.DataFrame(formula_feature_rows)
formula_coefficients = pd.DataFrame(
    np.atleast_2d(formula_ridge.coef_),
    index=list(ridge_manifest.target_columns),
    columns=formula_feature_table["feature"].tolist(),
)
formula_coefficients.insert(0, "intercept", np.atleast_1d(formula_ridge.intercept_))
formula_target_inverse = (
    "g⁻¹(z) = expm1(z) = exp(z) − 1" if ridge_manifest.selected_log1p else "g⁻¹(z) = z"
)
formula_input_transform = (
    "fⱼ(x) = log1p(x) for log1p-eligible water-level predictors;"
    if ridge_manifest.selected_log1p
    else "fⱼ(x) = x for every predictor;"
)
display(
    Markdown(
        f"""## Selected Ridge model formula

The saved model uses subset **{ridge_manifest.selected_subset}**, 
alpha **{ridge_manifest.selected_alpha:g}**, and 
log1p **{ridge_manifest.selected_log1p}**. For each forecast horizon 
h = 1, …, {len(ridge_manifest.target_columns)}, its prediction is:

**ŷ[t+h] = g⁻¹(b[h] + Σⱼ β[h,j] · ((fⱼ(x[t,j]) − μ[j]) / s[j]))**

Here, {formula_input_transform} and {formula_target_inverse}. 
The β[h,j] coefficients and b[h] intercepts are shown below; 
μ[j] and s[j] are the fitted StandardScaler center and scale.
"""
    )
)
display(formula_feature_table)
display(formula_coefficients)